# 04. Ecualización de histograma

**Objetivo:** comprender y construir paso a paso una ecualización global de histograma antes de compararla con OpenCV.

In [ ]:
import numpy as np
import cv2

from filtrado_digital.io import cargar_imagen, a_grises, ruta_imagen_ejemplo
from filtrado_digital.visualizacion import comparar, imagen_e_histograma
from filtrado_digital.histogramas import histograma_manual, ecualizar_histograma_manual

## 1. Preparar una imagen de bajo contraste

Comprimiremos las intensidades de la fotografía para observar con claridad el efecto de la ecualización.

In [ ]:
foto = cargar_imagen(ruta_imagen_ejemplo())
original = a_grises(foto)
bajo_contraste = (100 + original.astype(np.float32) * (70 / 255)).clip(0, 255).astype(np.uint8)
imagen_e_histograma(bajo_contraste, "Contraste reducido")

## 2. PDF, CDF y LUT

La ecualización utiliza la distribución de intensidades para construir una nueva asignación.

- **PDF:** proporción de píxeles en cada intensidad: `histograma / número_de_píxeles`.
- **CDF:** suma acumulada de la PDF; indica qué proporción de píxeles se encuentra hasta cada intensidad.
- **LUT:** tabla de consulta que asigna una intensidad de salida a cada intensidad de entrada.

El procedimiento es: **histograma → PDF → CDF → LUT → imagen ecualizada**.

In [ ]:
# Implementación explícita, paso a paso.
hist = histograma_manual(bajo_contraste)
pdf = hist / bajo_contraste.size
cdf = np.cumsum(pdf)

cdf_no_cero = cdf[cdf > 0]
cdf_min = cdf_no_cero[0]

lut = np.round((cdf - cdf_min) / (1 - cdf_min) * 255)
lut = np.clip(lut, 0, 255).astype(np.uint8)

ecualizada_paso_a_paso = lut[bajo_contraste]
comparar([bajo_contraste, ecualizada_paso_a_paso], ["Antes", "Ecualizada paso a paso"])

## 3. Función reutilizable y OpenCV

Ahora utilizamos la implementación reutilizable del proyecto y la versión optimizada de OpenCV para comprobar que el procedimiento anterior produce el resultado esperado.

In [ ]:
ecualizada_manual, lut_funcion = ecualizar_histograma_manual(bajo_contraste)
ecualizada_cv = cv2.equalizeHist(bajo_contraste)

comparar(
    [ecualizada_paso_a_paso, ecualizada_manual, ecualizada_cv],
    ["Paso a paso", "Función del proyecto", "OpenCV"],
)

print("Paso a paso = función:", np.array_equal(ecualizada_paso_a_paso, ecualizada_manual))
print(
    "Error medio absoluto manual/OpenCV:",
    np.mean(np.abs(ecualizada_manual.astype(np.int16) - ecualizada_cv.astype(np.int16)))
)

## 4. Limitación de la ecualización global

La misma transformación se aplica a toda la imagen. Cuando distintas regiones tienen condiciones de iluminación diferentes, una mejora global puede favorecer unas zonas y perjudicar otras. En el siguiente módulo estudiaremos CLAHE, que trabaja localmente.

## Conclusiones

- La ecualización puede construirse a partir de histograma, PDF, CDF y LUT.
- La LUT transforma cada intensidad original en una nueva intensidad.
- OpenCV implementa el mismo objetivo mediante `cv2.equalizeHist()`.
- La ecualización global no siempre es apropiada cuando el contraste cambia espacialmente.